# 03 Traditional Text Representations: One-Hot, BoW, N-Grams & TF-IDF

**Real-World Scenario**: Enterprise IT Infrastructure Failure Incident Retrieval System.

This notebook demonstrates building a Scikit-Learn `TfidfVectorizer` matrix, computing Cosine similarity search, saving vectorizer artifacts into a hidden `.model_cache/` directory, and executing a **Complete GenAI Incident Triage LLM Call**.

## Step 1: Environment Setup & Local Cache Configuration

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from root .env file
load_dotenv()

# Create hidden model cache directory for local pipeline artifact persistence
os.makedirs(".model_cache", exist_ok=True)

print("=== Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden Cache Directory Path:", os.path.abspath(".model_cache"))

## Step 2: Realistic IT Incident Logs Dataset Setup

In [ ]:
incidents = [
    "PostgreSQL primary database failover script failed due to connection timeout on port 5432.",
    "Microservice API gateway authentication failed with HTTP status 401 unauthorized token.",
    "PostgreSQL database backup completed successfully in 45 seconds.",
    "Kubernetes pod worker restarted due to Out Of Memory (OOM) killer event."
]

print(f"Loaded {len(incidents)} IT failure incident logs.")
print("Sample Incident #1:", incidents[0])

## Step 3: TF-IDF Feature Extraction & Matrix Construction

In [ ]:
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Initialize TfidfVectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    sublinear_tf=True,
    ngram_range=(1, 2),
    max_features=12
)

# 2. Transform incidents into sparse matrix
tfidf_matrix = vectorizer.fit_transform(incidents)

# 3. Save fitted vectorizer artifact to hidden folder
model_save_path = os.path.join(".model_cache", "tfidf_vectorizer.pkl")
joblib.dump(vectorizer, model_save_path)
print(f"Fitted TfidfVectorizer saved to hidden folder: {model_save_path}")

feature_names = vectorizer.get_feature_names_out()
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names, index=[f"Inc #{i+1}" for i in range(len(incidents))])
print("
=== Top 12 TF-IDF Feature Matrix ===")
print(df_tfidf.round(4))

## Step 4: Cosine Similarity Sparse Search Query Execution

In [ ]:
query = "database connection failover timeout"
query_vec = vectorizer.transform([query])
sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

ranked_indices = sim_scores.argsort()[::-1]

print(f"Search Query: '{query}'")
print("=== TF-IDF Cosine Similarity Search Rankings ===")
for rank_idx, idx in enumerate(ranked_indices, 1):
    print(f"Rank #{rank_idx} (Score: {sim_scores[idx]:.4f}) | {incidents[idx]}")

## Step 5: Executing Complete GenAI Incident Triage LLM Call

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

top_matched_doc = incidents[ranked_indices[0]]

if os.getenv("OPENAI_API_KEY"):
    prompt = ChatPromptTemplate.from_template('''You are an Enterprise Site Reliability Engineer AI.
Provide an incident triage report strictly using the retrieved incident context below.

Retrieved Context Log:
{context}

User Query: {query}

Triage Report:''')
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    
    response = llm.invoke(prompt.format(context=top_matched_doc, query=query))
    print("=== Complete GenAI Incident Triage Response ===")
    print(response.content)
else:
    print("[NOTICE] OPENAI_API_KEY not found in .env. Skipping live LLM call.")